#### 그라디오 챗봇과 랭체인 LLM 연동하기
* Groq / OpenAI 모델을 `get_llm()` 함수 하나로 만들고, Gradio `Chatbot`에 연결합니다.
* 이 노트북의 챗봇은 **대화 기록을 기억**하고, 응답을 **스트리밍**으로 보여 주며, 오류가 나도 화면이 멈추지 않습니다.

In [ ]:
# 필요한 라이브러리 설치 (프로젝트는 uv 사용)
# uv add gradio langchain langchain-openai python-dotenv

#### LLM 설정 (Groq / OpenAI 선택)

- **Groq**와 **OpenAI** 두 가지 모델을 모두 사용할 수 있습니다. (Groq는 OpenAI 호환 API라서 `ChatOpenAI`로 연결)
- `get_llm(provider="groq")`처럼 호출하면 제공자별 설정(키, 엔드포인트, 모델명)으로 `ChatOpenAI`를 만들어 줍니다. `provider`를 생략하면 아래 셀의 `PROVIDER` 값을 사용합니다.
- 사용할 제공자의 키(`GROQ_API_KEY` 또는 `OPENAI_API_KEY`)가 `.env`에 있어야 합니다. OpenAI를 선택하면 사용 요금이 발생합니다.

In [ ]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 " if key else "없음"
    print(f"{name}: {status}")

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — provider 인자를 생략했을 때 사용되는 기본값
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",  # 대안: "moonshotai/kimi-k2-instruct-0905"
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: ChatOpenAI 에 그대로 전달할 추가 옵션 (예: timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

#### 1. LLM 호출하기
* `llm.invoke(...)`는 `AIMessage`를 반환합니다. 답변 문자열은 `response.content`입니다.

In [ ]:
# LLM 인스턴스 생성 (제공자/키/모델명은 get_llm 이 설정)
llm = get_llm(temperature=0, provider="groq")

response = llm.invoke([{"role": "user", "content": "파이썬이란?"}])
print(response)          # AIMessage
print(response.content)  # str

#### 2. 제공자 바꾸기
* `provider="openai"`로 바꾸면 같은 코드로 OpenAI 모델을 사용합니다. (사용 요금 발생)
* API 키는 `get_llm` 안에서 `.env` 값을 명시적으로 전달하므로, 호출하는 코드에는 키가 나타나지 않습니다.

In [ ]:
llm = get_llm(temperature=0, provider="openai")   # OpenAI: get_llm(provider="openai")

response = llm.invoke([{"role": "user", "content": "Langchain이란?"}])
print(response.content)

In [ ]:
#### 3. Gradio 챗봇에 LLM 연결하기 ( 초기 버전 )


In [ ]:
# 그라디오 라이브러리를 불러옵니다.
import gradio as gr

llm = get_llm(temperature=0, provider="groq")

# 채팅봇의 응답을 처리하는 함수를 정의합니다.
def chat_respond(message, chat_history):  
    response = llm.invoke([{"role": "user", "content": message}])
    bot_message = response.content #str
    
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": bot_message})

    # 수정된 채팅 기록을 반환합니다.
    return "", chat_history  

# gr.Blocks()를 사용하여 인터페이스를 생성합니다.
with gr.Blocks() as demo:  
    chatbot = gr.Chatbot(label="채팅창")  # '채팅창'이라는 레이블을 가진 채팅봇 컴포넌트를 생성합니다.
    msg = gr.Textbox(label="입력")  # '입력'이라는 레이블을 가진 텍스트박스를 생성합니다.
    clear = gr.Button("초기화")  # '초기화'라는 레이블을 가진 버튼을 생성합니다.

    # 텍스트박스에 메시지를 입력하고 제출하면 respond 함수가 호출되도록 합니다.
    msg.submit(chat_respond, [msg, chatbot], [msg, chatbot])  
    # '초기화' 버튼을 클릭하면 채팅 기록을 초기화합니다.
    clear.click(lambda: None, None, chatbot, queue=False)  

# 인터페이스를 실행합니다. 실행하면 사용자는 '입력' 텍스트박스에 메시지를 작성하고 제출할 수 있으며, '초기화' 버튼을 통해 채팅 기록을 초기화 할 수 있습니다.
demo.launch(debug=True)  

#### 참고. `yield` 이해하기 
* `return`은 값을 **한 번** 돌려주고 함수를 끝냅니다.
* `yield`는 값을 **하나씩 내보내고 잠시 멈췄다가**, 다음 값을 요청받으면 이어서 실행합니다. `yield`가 들어간 함수를 **제너레이터(generator)** 라고 합니다.
* Gradio는 함수가 `yield`한 값을 받을 때마다 화면을 갱신합니다. 그래서 LLM 답변을 글자가 늘어나 듯 보여 줄 수 있습니다.

In [ ]:
# 예제 1. return 과 yield 의 차이
def with_return():
    """return: 값을 한 번 돌려주고 끝난다."""
    return [1, 2, 3]

def with_yield():
    """yield: 값을 하나씩 내보낸다 (제너레이터 함수)."""
    yield 1
    yield 2
    yield 3

print("return:", with_return())      # 리스트가 한 번에 나옴
gen = with_yield()                   # 호출만으로는 실행되지 않고 제너레이터 객체만 만들어짐
print("yield :", gen)
print("yield :", list(gen))          # 값을 모두 꺼내면 리스트와 같은 결과

In [ ]:
# 예제 2. 실행이 yield 에서 멈췄다가 이어지는 모습
def count_up(limit: int):
    """1부터 limit 까지 하나씩 내보낸다. yield 마다 실행이 멈춘다."""
    for n in range(1, limit + 1):
        print(f"  [함수] {n} 을(를) yield 하기 직전")
        yield n
        print(f"  [함수] {n} 을(를) 내보낸 뒤 이어서 실행")

for value in count_up(3):            # for 문이 값을 요청할 때마다 함수가 한 단계씩 실행됨
    print(f"[받는 쪽] 받은 값: {value}")

In [ ]:
# 예제 3. 스트리밍 흉내: 문장을 한 글자씩 내보내며 누적하기 (챗봇 답변이 늘어나는 원리)
import time

STREAM_DELAY = 0.05   # 글자 사이 대기 시간(초) - LLM 이 조각을 보내는 간격을 흉내냄

def fake_stream(sentence: str):
    """문장을 한 글자씩 yield 한다. (llm.stream() 이 조각을 보내는 것과 같은 모양)"""
    for ch in sentence:
        time.sleep(STREAM_DELAY)
        yield ch

def fake_chat(message: str):
    """지금까지 받은 글자를 누적해서 매번 yield 한다. (chat_respond 의 yield 와 같은 패턴)"""
    partial = ""
    for ch in fake_stream(f"'{message}' 라고 말씀하셨네요!"):
        partial += ch          # 새 조각을 이어 붙이고
        yield partial          # 지금까지의 전체 답변을 내보낸다 -> 화면에서는 글자가 늘어나는 것처럼 보임

for step in fake_chat("안녕"):
    print(step)                # Gradio 라면 이 값으로 채팅창을 갱신한다

* 예제 3의 `fake_chat`이 4번 챗봇의 `chat_respond`와 같은 구조입니다. `partial += 조각` 후 `yield`하면, 받는 쪽(Gradio)이 매번 **지금까지의 전체 답변**을 받아 화면을 덮어씁니다.
* 제너레이터는 **한 번 다 꺼내면 끝**입니다. 다시 쓰려면 함수를 다시 호출해 새 제너레이터를 만들어야 합니다.

#### 4. Gradio 챗봇에 LLM 연결하기 ( advanced 버전 스트리밍 방식 )

| 개선 항목 | 설명 |
|---|---|
| **대화 기록 전달** | 이전 대화를 LLM에 함께 보내 맥락을 기억합니다. (최근 `MAX_HISTORY_MESSAGES`개만 전달해 토큰/비용 절약) |
| **스트리밍 응답** | `llm.stream()`과 `yield`로 답변을 생성되는 대로 표시합니다. |
| **오류 처리** | API 오류가 나도 앱이 멈추지 않고, 채팅창에 오류 메시지를 표시합니다. (오류 메시지는 LLM 입력에서 제외) |
| **빈 입력 무시** | 공백만 입력하면 아무 일도 하지 않습니다. |
| **시스템 프롬프트** | 한국어로 답하고 모르면 모른다고 하도록 지시합니다. |
| **초기화 버튼** | 채팅 기록과 입력창을 함께 비웁니다. |
| **포트 해제** | `debug=True`로 노트북을 막지 않고, `close_demo()`로 서버를 종료합니다. |

* Gradio 6의 `Chatbot`은 기록의 `content`를 `[{"type": "text", "text": "..."}]` 리스트로 전달합니다. 그래서 `to_text()`로 문자열로 변환합니다.

##### 4-1. 서버 실행/종료 헬퍼 함수

| 함수 | 하는 일 | 언제 쓰나요? |
|---|---|---|
| `launch_demo(demo)` | 켜져 있는 서버를 먼저 끄고, `demo`를 새로 실행합니다. | 챗봇을 띄울 때. 셀을 다시 실행해도 포트 충돌이 없습니다. |
| `close_demo()` | 실행 중인 Gradio 서버를 모두 끕니다. | 실습을 마칠 때. 사용하던 포트(7860)가 반납됩니다. |
| `is_port_open()` | 포트를 누군가 쓰고 있으면 `True`, 아니면 `False`를 반환합니다. | 서버가 꺼졌는지 확인할 때. |

In [ ]:
import socket
import gradio as gr

GRADIO_HOST = "127.0.0.1"   # 내 컴퓨터에서만 접속 가능한 주소 (외부 접속 차단)
GRADIO_PORT = 7860          # 웹 서버가 사용할 포트 번호 (브라우저 주소: http://127.0.0.1:7860)

def is_port_open(port: int = GRADIO_PORT, host: str = GRADIO_HOST) -> bool:
    """해당 포트에서 서버가 응답하면 True(점유 중), 아니면 False(해제됨)를 반환한다."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)                       # 응답이 없으면 0.5초 뒤 포기
        return s.connect_ex((host, port)) == 0  # 접속 성공(0)이면 누군가 포트를 쓰는 중

def close_demo() -> None:
    """실행 중인 모든 Gradio 서버를 종료하여 포트를 해제한다."""
    gr.close_all()

def launch_demo(demo: gr.Blocks, port: int = GRADIO_PORT) -> None:
    """이전 서버를 종료한 뒤 demo를 실행한다. 셀을 다시 실행해도 포트 충돌이 없다."""
    close_demo()  # 이미 켜진 서버가 있으면 먼저 끈다 (포트 충돌 방지)
    # prevent_thread_lock=True: 서버를 켠 뒤 셀 실행을 끝낸다 (False면 서버가 꺼질 때까지 셀이 멈춤)
    demo.launch(server_name=GRADIO_HOST, server_port=port, prevent_thread_lock=True)

##### 3-2. 챗봇 코드 한눈에 보기

**전체 흐름** — 사용자가 메시지를 입력하고 Enter를 누르면 일어나는 일입니다.

```text
[입력창에 메시지 입력 + Enter]
        │  msg.submit 이벤트
        ▼
chat_respond(message, chat_history)
   ① 빈 입력이면 아무것도 하지 않고 종료
   ② build_messages()  : 시스템 프롬프트 + 이전 대화 + 현재 질문  →  LLM에 보낼 메시지 목록
   ③ 사용자 메시지를 채팅창에 즉시 표시          (yield)
   ④ chat_llm.stream() : LLM이 답을 한 조각씩 보내줌
   ⑤ 조각이 올 때마다 채팅창의 답변을 갱신       (yield)  → 글자가 타이핑되는 것처럼 보임
   ⑥ 오류가 나면 채팅창에 ⚠️ 안내 메시지 표시
```

**함수 요약**

| 함수 | 한 줄 설명 | 입력 → 출력 |
|---|---|---|
| `to_text(content)` | Gradio가 주는 `content`를 **문자열**로 바꿉니다. (Gradio 6은 `[{"type": "text", "text": "..."}]` 리스트로 줌) | `content` → `str` |
| `build_messages(chat_history, message)` | 시스템 프롬프트 + 최근 대화 + 현재 질문을 LangChain 메시지 목록으로 만듭니다. **LLM이 이전 대화를 기억하게 하는 핵심**입니다. | 대화 기록, 질문 → 메시지 `list` |
| `chat_respond(message, chat_history)` | 챗봇의 **심장**. 질문을 받아 LLM 답변을 스트리밍으로 만들어 채팅창을 갱신합니다. | 질문, 대화 기록 → (`yield`) 빈 입력창, 갱신된 대화 기록 |
| `build_chat_ui()` | 채팅창, 입력창, 초기화 버튼을 배치하고 **이벤트를 연결**합니다. | 없음 → `gr.Blocks` |

**알아두면 좋은 개념**

* **`chat_history` (대화 기록)**: 역할(`role`)과 내용(`content`)을 가진 딕셔너리의 리스트입니다. `user`는 사용자, `assistant`는 챗봇입니다.

  ```python
  [
      {"role": "user",      "content": "내 이름은 철수야"},
      {"role": "assistant", "content": "안녕하세요 철수님!"},
  ]
  ```
* **`yield` (제너레이터)**: `return`은 함수를 끝내며 값을 한 번만 돌려주지만, `yield`는 값을 **여러 번** 내보냅니다. Gradio는 `yield`된 값을 받을 때마다 화면을 갱신하므로, 답변이 생성되는 대로 보여 줄 수 있습니다.
* **스트리밍(`stream`)**: `invoke()`는 답변이 **전부 완성된 뒤** 한 번에 돌려주고, `stream()`은 **조각(chunk)** 단위로 돌려줍니다.
* **이벤트 연결**: `msg.submit(함수, 입력들, 출력들)`은 "입력창에서 Enter를 누르면 `함수(입력들)`을 실행하고, 반환값을 `출력들`에 넣어라"는 뜻입니다. `inputs=[msg, chatbot]`의 순서가 함수 인자 `(message, chat_history)`의 순서와 같습니다.
* **상수**: `SYSTEM_PROMPT`(챗봇의 성격과 규칙), `MAX_HISTORY_MESSAGES`(LLM에 보낼 최근 대화 수, 많을수록 비용 증가), `ERROR_PREFIX`(오류 메시지 구분용 표시)

##### 참고. `build_messages()`가 하는 일 (Gradio 형식 → LangChain 형식)

LLM은 **이전 대화를 스스로 기억하지 못합니다.** 그래서 매번 "지금까지의 대화 전체"를 다시 보내야 하고, `build_messages()`가 그 목록을 만듭니다. 즉 **Gradio가 주는 형식(dict)을 LangChain이 쓰는 형식(메시지 객체)으로 바꾸는 번역기**입니다.

| 변수 | 타입 | 예시 |
|---|---|---|
| `chat_history` | `list` (dict의 리스트) | `[{...}, {...}]` |
| `item` | `dict` (Gradio가 주는 메시지 1건) — `AIMessage`가 **아닙니다** | `{"role": "user", "content": "안녕"}` |
| `text` | `str` | `"안녕"` |
| `messages`의 원소 | `SystemMessage` / `HumanMessage` / `AIMessage` | `item`을 변환해 만든 결과물 |

**변환 예시**

```python
# 입력: chat_history (Gradio 형식, dict)
[
    {"role": "user",      "content": "내 이름은 철수야"},
    {"role": "assistant", "content": "안녕하세요 철수님!"},
]

# message = "내 이름이 뭐야?" 일 때 출력: messages (LangChain 형식, 객체)
[
    SystemMessage(content="당신은 친절하고 ..."),   # 1) 시스템 프롬프트
    HumanMessage(content="내 이름은 철수야"),        # 2) 과거 user      -> Human
    AIMessage(content="안녕하세요 철수님!"),         # 2) 과거 assistant -> AI
    HumanMessage(content="내 이름이 뭐야?"),         # 3) 현재 질문
]
```

##### 참고. `msg.submit()`은 무엇인가요? (이벤트 연결)

```python
msg.submit(chat_respond, inputs=[msg, chatbot], outputs=[msg, chatbot])
```

* `msg`는 `gr.Textbox` 컴포넌트이고, `submit`은 이 컴포넌트에 **이벤트 리스너(event listener)를 등록하는 메서드**입니다.
* `inputs`, `outputs`는 `submit()`의 **매개변수(parameter)** 이며, 키워드 인자로 넘긴 것입니다. 위치 인자로 `msg.submit(chat_respond, [msg, chatbot], [msg, chatbot])`처럼 써도 같습니다.

```python
msg.submit(fn, inputs, outputs, ...)
#          │    │       └ 반환값(yield 값)을 넣을 컴포넌트들
#          │    └ 함수에 전달할 값을 꺼낼 컴포넌트들
#          └ 이벤트가 발생했을 때 실행할 함수
```

**의미**: "입력창에서 Enter를 누르면(`submit` 이벤트), `inputs` 컴포넌트의 **현재 값**을 읽어 `chat_respond`에 넘기고, 반환값(또는 `yield` 값)을 `outputs` 컴포넌트에 **넣어라**."

| 구분 | 값 | 동작 |
|---|---|---|
| `fn` | `chat_respond` | 실행할 함수 |
| `inputs=[msg, chatbot]` | 입력창 값, 채팅창 대화 기록 | 이 순서대로 `chat_respond(message, chat_history)`의 인자가 됩니다. |
| `outputs=[msg, chatbot]` | 입력창, 채팅창 | `yield "", chat_history`의 첫 값은 `msg`, 둘째 값은 `chatbot`에 들어갑니다. |

**핵심 규칙: 순서 대응**

```python
def chat_respond(message, chat_history):      # 인자 순서
    ...
    yield "", chat_history                    # 반환 순서

msg.submit(chat_respond, inputs=[msg, chatbot], outputs=[msg, chatbot])
#                               ↑   ↑                   ↑   ↑
#                  message ←────┘   └─ chat_history     │   └─ 두 번째 값(대화 기록)이 들어감
#                                                       └─ 첫 번째 값("")이 들어감 -> 입력창이 비워짐
```

* `inputs` 리스트의 **순서와 개수**는 함수 인자와 같아야 합니다.
* `outputs` 리스트의 **순서와 개수**는 함수가 내보내는 값과 같아야 합니다.
* `msg`가 `outputs`에도 들어 있어서, `""`를 내보내면 입력창이 비워집니다.
* 같은 방식으로 `clear.click(lambda: ([], ""), inputs=None, outputs=[chatbot, msg], queue=False)`도 읽을 수 있습니다. 버튼 클릭에는 입력이 필요 없어서 `None`이고, 반환값 `([], "")`이 `chatbot`에 빈 리스트, `msg`에 빈 문자열로 들어갑니다.

**Q. `msg`는 Textbox, `chatbot`은 Chatbot인데 `str`, `list` 타입 인자로 어떻게 전달되나요?**

* `inputs=[msg, chatbot]`의 `msg`, `chatbot`은 **"어느 컴포넌트에서 값을 읽을지 가리키는 표지판"** 입니다. 함수에는 컴포넌트 객체가 아니라 **컴포넌트의 현재 값**이 전달되므로 타입이 맞습니다.
* 각 컴포넌트가 "화면 값 ↔ 파이썬 값" 변환 규칙을 갖고 있고, Gradio가 대신 처리합니다.

| 컴포넌트 | 클래스 | 함수로 전달되는 값 | 타입 |
|---|---|---|---|
| `msg` | `gr.Textbox` | 입력창에 적힌 글자 | `str` |
| `chatbot` | `gr.Chatbot` | 현재 대화 기록 | `list` (dict의 리스트) |

```text
브라우저: 입력창 "안녕", 채팅창 대화 기록 [..]  (JSON 데이터)
   │  ① 브라우저가 현재 값들을 서버로 전송
   ▼
Gradio 서버
   │  ② 각 컴포넌트가 값을 파이썬 타입으로 변환
   │     Textbox.preprocess(...) -> str
   │     Chatbot.preprocess(...) -> list
   │  ③ inputs 순서대로 함수 인자에 대입
   ▼
chat_respond("안녕", [..])
   │  ④ yield 값 ("", chat_history)
   │     Textbox.postprocess("")   -> 화면용 값
   │     Chatbot.postprocess([..]) -> 화면용 값
   ▼
브라우저: 입력창 비워짐, 채팅창 갱신
```

* 입력 방향: 화면 값 → `preprocess` → 파이썬 값 → 함수 인자
* 출력 방향: 함수 반환값 → `postprocess` → 화면 값 → 컴포넌트
* `outputs=[msg, chatbot]`도 같은 원리입니다. `yield "", chat_history`의 `""`(`str`)는 **Textbox의 값**으로, `chat_history`(`list`)는 **Chatbot의 값**으로 들어갑니다. 컴포넌트 객체가 아니라 **값만** 반환합니다.
* 연결은 이름이 아니라 **순서**로 합니다. `msg`가 `message`로 가는 것은 이름이 비슷해서가 아니라 `inputs` 리스트의 첫 번째이기 때문입니다.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ── 설정 상수 ────────────────────────────────────────────────────────────
# 시스템 프롬프트: 챗봇의 성격과 규칙을 정하는 지시문 (대화 맨 앞에 항상 들어간다)
SYSTEM_PROMPT = "당신은 친절하고 정확한 한국어 AI 어시스턴트입니다. 모르는 내용은 추측하지 말고 모른다고 답하세요."
MAX_HISTORY_MESSAGES = 10   # LLM에 전달할 최근 대화 메시지 수 (사용자+AI 합산, 많을수록 토큰/비용 증가)
ERROR_PREFIX = "⚠️"         # 오류 메시지 표시용 접두어 (이 접두어로 시작하는 메시지는 LLM 입력에서 제외)

# 챗봇용 LLM (제공자/키/모델명은 get_llm 이 설정)
chat_llm = get_llm(temperature=0, provider="groq")


# ── 변환 함수: Gradio의 content -> 문자열 ─────────────────────────────
def to_text(content) -> str:
    """Gradio 6 Chatbot의 content(str 또는 [{"type": "text", "text": ...}] 리스트)를 문자열로 변환한다."""
    if isinstance(content, str):      # 이미 문자열이면 그대로 사용
        return content
    if isinstance(content, list):     # 리스트면 각 조각의 "text"를 이어 붙인다
        return "".join(part.get("text", "") for part in content if isinstance(part, dict))
    return str(content or "")         # None 등 그 밖의 값은 빈 문자열/문자열로 처리


# ── LLM 입력 만들기: 이전 대화를 기억하게 하는 핵심 ────────────────────
def build_messages(chat_history: list, message: str) -> list:
    """시스템 프롬프트 + 최근 대화 기록 + 현재 메시지를 LangChain 메시지 리스트로 만든다."""
    # 1. 시스템 프롬프트 넣기: 목록 맨 앞에 "너는 이런 챗봇이야"라는 지시문(`SystemMessage`)을 넣습니다.
    messages = [SystemMessage(content=SYSTEM_PROMPT)]            
    # 2. 과거 대화 변환: `chat_history[-MAX_HISTORY_MESSAGES:]`로 최근 N개만 하나씩 꺼냅니다.
    for item in chat_history[-MAX_HISTORY_MESSAGES:]:  
        # `to_text(item["content"])`: Gradio 6은 `content`를 리스트로 줄 때가 있어 문자열로 통일합니다.          
        text = to_text(item["content"])
        # 빈 답변이나 `⚠️` 오류 메시지는 건너뜁니다.
        if not text or text.startswith(ERROR_PREFIX):            
            continue
        # role 이 `"user"`면 `HumanMessage`로 역할에 맞는 객체를 만들어 붙입니다.
        if item["role"] == "user":
            messages.append(HumanMessage(content=text))          
         # role 이 `"assistant"`면 `AIMessage`로 역할에 맞는 객체를 만들어 붙입니다.  
        elif item["role"] == "assistant":
            messages.append(AIMessage(content=text))
    # 현재 질문 붙이기: 지금 입력한 질문은 아직 `chat_history`에 없으므로 맨 뒤에 따로 `HumanMessage`로 추가합니다.                     
    messages.append(HumanMessage(content=message))               
    return messages

# ── 챗봇의 심장: 질문을 받아 답변을 스트리밍으로 생성 ──────────────────
def chat_respond(message: str, chat_history: list):
    """사용자 메시지에 대한 LLM 응답을 스트리밍으로 생성한다. (Gradio가 yield 값을 순서대로 화면에 반영)

    yield 하는 값은 (입력창에 넣을 값, 채팅창에 넣을 대화 기록) 한 쌍이며,
    msg.submit 의 outputs=[msg, chatbot] 순서와 같다.
    """
    chat_history = chat_history or []             # 처음에는 None 일 수 있으므로 빈 리스트로 대체
    if not message or not message.strip():        # ① [빈 입력 방어] 공백만 입력하면 아무것도 하지 않음
        yield "", chat_history
        return

    messages = build_messages(chat_history, message)   # ② 화면에 추가하기 "전"의 기록으로 LLM 입력을 만든다
    chat_history = chat_history + [                    # 기존 리스트를 수정하지 않고 새 리스트를 만든다
        {"role": "user", "content": message},          #   방금 보낸 사용자 메시지
        {"role": "assistant", "content": ""},          #   챗봇 답변 자리 (아직 비어 있음)
    ]
    yield "", chat_history                        # ③ 사용자 메시지를 먼저 표시하고, 입력창("")을 비운다

    partial = ""                                  # 지금까지 받은 답변 조각을 모으는 변수
    try:
        for chunk in chat_llm.stream(messages):   # ④ LLM이 답변 조각(chunk)을 하나씩 보내준다
            partial += to_text(chunk.content)     # 새 조각을 이어 붙이고
            chat_history[-1]["content"] = partial #   마지막 메시지(챗봇 답변 자리)를 갱신한 뒤
            yield "", chat_history                # ⑤ 화면에 반영 -> 글자가 타이핑 되듯 늘어난다
    except Exception as e:                        # ⑥ [오류 방어] API 오류가 나도 앱은 멈추지 않는다
        chat_history[-1]["content"] = f"{ERROR_PREFIX} 응답 생성 중 오류가 발생했습니다: {e}"
        yield "", chat_history


# ── 화면 구성 + 이벤트 연결 ───────────────────────────────────────────
def build_chat_ui() -> gr.Blocks:
    """Chatbot + Textbox + 초기화 버튼으로 구성된 LLM 챗봇 UI를 만든다."""
    with gr.Blocks() as demo:                     # Blocks: 컴포넌트를 위에서 아래로 쌓는 화면 틀
        gr.Markdown("## LLM 챗봇")                 # 제목
        chatbot = gr.Chatbot(label="채팅창", height=450)   # 대화가 표시되는 영역
        msg = gr.Textbox(label="입력", placeholder="메시지를 입력하고 Enter를 누르세요.", autofocus=True)  # 입력창
        clear = gr.Button("초기화")                # 대화 초기화 버튼

        # 이벤트 연결: 입력창에서 Enter(submit) -> chat_respond(msg, chatbot) 실행
        #   -> yield 된 값이 [msg, chatbot]에 차례로 들어간다 (입력창 비움 + 대화 갱신)
        msg.submit(chat_respond, inputs=[msg, chatbot], outputs=[msg, chatbot])
        # 이벤트 연결: 초기화 버튼 클릭 -> 채팅창은 빈 리스트([]), 입력창은 빈 문자열("")로
        #   queue=False: 대기열 없이 즉시 실행
        clear.click(lambda: ([], ""), inputs=None, outputs=[chatbot, msg], queue=False)
    return demo


# ── 실행: 화면을 만들어 서버를 켠다 (종료는 아래 close_demo() 셀) ────────
launch_demo(build_chat_ui())

In [ ]:
close_demo()
print("포트 점유 여부:", is_port_open())   # False면 해제 완료